Imports

In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import copy
from torchsummary import summary

Dataset Paths for OUR MASTER dataset

In [32]:
train_dir = r"C:\SUTD\50.021 Artificial Intelligence\pppppp\Master_Dataset\train"
val_dir = r"C:\SUTD\50.021 Artificial Intelligence\pppppp\Master_Dataset\val"
test_dir = r"C:\SUTD\50.021 Artificial Intelligence\pppppp\Master_Dataset\test"

In [33]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print(f'Using GPU: {torch.cuda.get_device_name(0)}')
else:
    print('Using CPU')

Using GPU: NVIDIA GeForce RTX 4060 Laptop GPU


Transforms

In [34]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


DataLoaders

In [35]:
# Load datasets
train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)
test_dataset = datasets.ImageFolder(test_dir, transform=transform)

print("Classes:", train_dataset.classes)
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print("Train class_to_idx:", train_dataset.class_to_idx)
print("Val class_to_idx:", val_dataset.class_to_idx)
print("Test class_to_idx:", test_dataset.class_to_idx)

Classes: ['NORMAL', 'PNEUMONIA']
Train size: 14054
Val size: 1757
Test size: 1757
Train class_to_idx: {'NORMAL': 0, 'PNEUMONIA': 1}
Val class_to_idx: {'NORMAL': 0, 'PNEUMONIA': 1}
Test class_to_idx: {'NORMAL': 0, 'PNEUMONIA': 1}


In [36]:
train_dir = r"C:\SUTD\50.021 Artificial Intelligence\pppppp\Master_Dataset\train"
val_dir = r"C:\SUTD\50.021 Artificial Intelligence\pppppp\Master_Dataset\val"
test_dir = r"C:\SUTD\50.021 Artificial Intelligence\pppppp\Pretrained_Model_Dataset\test"

In [37]:
# Load datasets
train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)
test_dataset = datasets.ImageFolder(test_dir, transform=transform)

print("Classes:", train_dataset.classes)
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print("Train class_to_idx:", train_dataset.class_to_idx)
print("Val class_to_idx:", val_dataset.class_to_idx)
print("Test class_to_idx:", test_dataset.class_to_idx)

Classes: ['NORMAL', 'PNEUMONIA']
Train size: 14054
Val size: 1757
Test size: 624
Train class_to_idx: {'NORMAL': 0, 'PNEUMONIA': 1}
Val class_to_idx: {'NORMAL': 0, 'PNEUMONIA': 1}
Test class_to_idx: {'NORMAL': 0, 'PNEUMONIA': 1}


In [38]:
# Sanity check one sample
sample, label = train_dataset[0]
print("Sample shape:", sample.shape)
print("Label index:", label)
print("Class name:", train_dataset.classes[label])

Sample shape: torch.Size([3, 224, 224])
Label index: 0
Class name: NORMAL


Evaluation Function for All Models

In [39]:
def evaluate_model(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)

            # In eval mode, GoogLeNet usually returns main logits only
            if hasattr(outputs, "logits"):
                outputs = outputs.logits
            elif isinstance(outputs, tuple):
                outputs = outputs[0]

            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)

            probs = torch.sigmoid(outputs).squeeze(1)
            preds = (probs >= 0.5).long()

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)

    return epoch_loss, acc, f1, auc

GOOGLENET MODEL

In [40]:
def build_googlenet():
    model = models.googlenet(weights=None, aux_logits=False)
    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, 512),
        nn.Dropout(0.3),
        nn.ReLU(inplace=True),
        nn.Linear(512, 128),
        nn.ReLU(inplace=True),
        nn.Linear(128, 32),
        nn.ReLU(inplace=True),
        nn.Linear(32, 2)
    )
    return model

ALEXNET MODEL

In [41]:
def build_alexnet():
    model = models.alexnet(weights=None)
    model.classifier = nn.Sequential(
        nn.Dropout(),
        nn.Linear(9216, 4096),
        nn.ReLU(inplace=True),
        nn.Dropout(),
        nn.Linear(4096, 1024),
        nn.ReLU(inplace=True),
        nn.Linear(1024, 512),
        nn.ReLU(inplace=True),
        nn.Linear(512, 128),
        nn.ReLU(inplace=True),
        nn.Linear(128, 32),
        nn.ReLU(inplace=True),
        nn.Linear(32, 2)
    )
    return model

RESNET18 MODEL

In [42]:
def build_resnet18():
    model = models.resnet18(weights=None)
    model.fc = nn.Sequential(
        nn.Dropout(),
        nn.Linear(512, 128),
        nn.ReLU(inplace=True),
        nn.Linear(128, 32),
        nn.ReLU(inplace=True),
        nn.Linear(32, 2),
        nn.ReLU(inplace=True)
    )
    return model

Load Models

In [43]:
def load_model(model_name, path, device):
    if model_name == "googlenet":
        model = build_googlenet()
    elif model_name == "alexnet":
        model = build_alexnet()
    elif model_name == "resnet18":
        model = build_resnet18()
    else:
        raise ValueError("Unknown model name")

    state_dict = torch.load(path, map_location=device)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    return model

In [44]:
criterion = nn.CrossEntropyLoss()

googlenet_model = load_model("googlenet", "googlenet_finetuned_baseline.pth", device)
alexnet_model = load_model("alexnet", "alexnet_finetuned_baseline.pth", device)
resnet_model = load_model("resnet18", "resnet18_finetuned_baseline.pth", device)

C:\Users\thoai\AppData\Local\Temp\ipykernel_35720\1341042654.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path, map_location=device)


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


Evaluate Googlenet

In [ ]:
test_loss, test_acc, test_f1, test_auc = evaluate_model(googlenet_model, test_loader, criterion, device)
print("\nFinal Test Results (Clean Data)")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")
print(f"Test AUC-ROC:  {test_auc:.4f}")

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


Evaluate Alexnet

In [ ]:
test_loss, test_acc, test_f1, test_auc = evaluate_model(alexnet_model, test_loader, criterion, device)
print("\nFinal Test Results (Clean Data)")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")
print(f"Test AUC-ROC:  {test_auc:.4f}")


Final Test Results (Clean Data)
Test Loss:     0.9505
Test Accuracy: 0.8205
Test F1 Score: 0.8744
Test AUC-ROC:  0.9668


Evaluate Resnet-18

In [ ]:
test_loss, test_acc, test_f1, test_auc = evaluate_model(resnet_model, test_loader, criterion, device)
print("\nFinal Test Results (Clean Data)")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")
print(f"Test AUC-ROC:  {test_auc:.4f}")


Final Test Results (Clean Data)
Test Loss:     0.4186
Test Accuracy: 0.8253
Test F1 Score: 0.8751
Test AUC-ROC:  0.9527


Dataset paths for nih full

In [ ]:
train_dir = r"C:\SUTD\50.021 Artificial Intelligence\pppppp\dataset\train"
val_dir = r"C:\SUTD\50.021 Artificial Intelligence\pppppp\dataset\val"
test_dir = r"C:\SUTD\50.021 Artificial Intelligence\pppppp\dataset\test"

In [ ]:
# Load datasets
train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)
test_dataset = datasets.ImageFolder(test_dir, transform=transform)

print("Classes:", train_dataset.classes)
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print("Train class_to_idx:", train_dataset.class_to_idx)
print("Val class_to_idx:", val_dataset.class_to_idx)
print("Test class_to_idx:", test_dataset.class_to_idx)

Googlenet on nih

In [ ]:
test_loss, test_acc, test_f1, test_auc = evaluate_model(model, test_loader, criterion, device)
print("\nFinal Test Results (Clean Data)")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")
print(f"Test AUC-ROC:  {test_auc:.4f}")

Alex on nih

In [ ]:
test_loss, test_acc, test_f1, test_auc = evaluate_model(alexnet_model, test_loader, criterion, device)
print("\nFinal Test Results (Clean Data)")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")
print(f"Test AUC-ROC:  {test_auc:.4f}")

resnet on nih

In [ ]:
test_loss, test_acc, test_f1, test_auc = evaluate_model(resnet_model, test_loader, criterion, device)
print("\nFinal Test Results (Clean Data)")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")
print(f"Test AUC-ROC:  {test_auc:.4f}")